In [22]:
from time import sleep

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

pd.set_option("display.max_columns", None)

firefox_options = Options()
firefox_options.add_argument("--headless")

URL_MAIN = "https://www.flashscore.com"
TOURNEY_LIST = ["indian-wells-2023"]


def get_text_or_empty(browser, by, value):
    elements = browser.find_elements(by, value)
    return elements[0].text if elements else ""


def scrape_match(match_id, tourney, browser):
    match_flashscore_id = match_id.get_attribute("id")[4:]
    print("Opening match:", match_id.get_attribute("id"))

    match_page = f"{URL_MAIN}/match/{match_flashscore_id}/"
    browser.get(match_page)
    sleep(1.5)

    stats_links = browser.find_elements(By.PARTIAL_LINK_TEXT, "STATS")

    if not stats_links:
        print("No stats link found")
        return None

    stats_url = stats_links[0].get_attribute("href")
    browser.get(stats_url)
    sleep(2.5)

    stat_rows = browser.find_elements(By.XPATH, "//div[@data-testid='wcl-statistics']")
    print("Stat rows found:", len(stat_rows))

    if not stat_rows:
        return None

    row_data = {}

    for row in stat_rows:
        values = row.find_elements(By.XPATH, ".//div[@data-testid='wcl-statistics-value']")
        category = row.find_element(By.XPATH, ".//div[@data-testid='wcl-statistics-category']").text

        if len(values) >= 2:
            row_data[f"{category}_left"] = values[0].text
            row_data[f"{category}_right"] = values[1].text

    player_links = browser.find_elements(By.CLASS_NAME, "participant__participantLink")
    players = [p.get_attribute("href").split("/")[4].strip() for p in player_links]

    row_data["player_left"] = players[0] if len(players) > 0 else ""
    row_data["player_right"] = players[1] if len(players) > 1 else ""
    row_data["date"] = get_text_or_empty(browser, By.CLASS_NAME, "duelParticipant__startTime")
    row_data["match_result"] = get_text_or_empty(browser, By.CLASS_NAME, "duelParticipant__score")
    row_data["match_info"] = tourney

    odds_elements = browser.find_elements(By.CLASS_NAME, "oddsValue")
    odds = [o.text for o in odds_elements[:2]]

    row_data["odds_left"] = odds[0] if len(odds) > 0 else ""
    row_data["odds_right"] = odds[1] if len(odds) > 1 else ""

    return row_data


for tourney in TOURNEY_LIST:
    tourney_results = f"{URL_MAIN}/tennis/atp-singles/{tourney}/results/"
    filename = f"{tourney}.csv"

    browser_results = webdriver.Firefox(options=firefox_options)

    try:
        browser_results.get(tourney_results)
        sleep(2.5)

        matches = browser_results.find_elements(By.CLASS_NAME, "event__match")

        print(filename)
        print("Matches found:", len(matches))

        matches_data = []

        browser_match = webdriver.Firefox(options=firefox_options)

        try:
            for match_id in matches:
                try:
                    match_data = scrape_match(match_id, tourney, browser_match)

                    if match_data is not None:
                        matches_data.append(match_data)
                except Exception as e:
                    print("Match failed:", match_id.get_attribute("id"), type(e).__name__, str(e)[:200])
                    continue

        finally:
            browser_match.close()

        print("Rows collected:", len(matches_data))

        if matches_data:
            df = pd.DataFrame(matches_data)
            df.to_csv(filename, index=False)
            print("CSV written:", filename)
            display(df)
        else:
            print("No match data collected; CSV not written.")

    finally:
        browser_results.close()

indian-wells-2023.csv
Matches found: 131
Opening match: g_2_xQwqr8Fj
Stat rows found: 18
Opening match: g_2_pfNlpBQ5
Stat rows found: 18
Opening match: g_2_WhUAQL1K
Stat rows found: 18
Opening match: g_2_bXXQWdJg
Stat rows found: 18
Opening match: g_2_l8L5o2Xg
Stat rows found: 18
Opening match: g_2_E9m9Snae
Stat rows found: 18
Opening match: g_2_dteUCDjl
Stat rows found: 18
Opening match: g_2_ljEywjcl
Stat rows found: 18
Opening match: g_2_CzQWN8TG
Stat rows found: 18
Opening match: g_2_jRnkCQne
Stat rows found: 18
Opening match: g_2_Y3ZVDFaC
Stat rows found: 18
Opening match: g_2_88zoqD0I
Stat rows found: 18
Opening match: g_2_nmXJGHFg
Stat rows found: 18
Opening match: g_2_QwWNFyVa
Stat rows found: 18
Opening match: g_2_v7ul3bof
Stat rows found: 18
Opening match: g_2_rFhWaXwg
Stat rows found: 18
Opening match: g_2_YPW6J3Ih
Stat rows found: 18
Opening match: g_2_xGbesSRd
Stat rows found: 18
Opening match: g_2_lIGmKeQ0
Stat rows found: 18
Opening match: g_2_0x2h0tIU
Stat rows found: 18

,Aces_left,Aces_right,Double Faults_left,Double Faults_right,1st serve percentage_left,1st serve percentage_right,1st serve points won_left,1st serve points won_right,2nd serve points won_left,2nd serve points won_right,Break Points Saved_left,Break Points Saved_right,1st return points won_left,1st return points won_right,2nd return points won_left,2nd return points won_right,Break Points Converted_left,Break Points Converted_right,Winners_left,Winners_right,Unforced errors_left,Unforced errors_right,Net points won_left,Net points won_right,Service Points Won_left,Service Points Won_right,Return Points Won_left,Return Points Won_right,Total Points Won_left,Total Points Won_right,Service games won_left,Service games won_right,Return games won_left,Return games won_right,Total games won_left,Total games won_right,player_left,player_right,date,match_result,match_info,odds_left,odds_right
0,1,0,0,2,76%,65%,81%\n(30/37),61%\n(19/31),58%\n(7/12),41%\n(7/17),0/0,0/3,39%\n(12/31),19%\n(7/37),59%\n(10/17),42%\n(5/12),3/3,0/0,19,4,10,14,77%\n(10/13),53%\n(8/15),76%\n(37/49),54%\n(26/48),46%\n(22/48),24%\n(12/49),61%\n(59/97),39%\n(38/97),100%\n(9/9),63%\n(5/8),38%\n(3/8),0%\n(0/9),71%\n(12/17),29%\n(5/17),alcaraz-garfia-carlos,medvedev-daniil,19.03.2023 23:10,2\n-\n0\nFINISHED,indian-wells-2023,,
1,5,4,1,4,72%,50%,73%\n(38/52),78%\n(28/36),45%\n(9/20),47%\n(17/36),1/2,4/6,22%\n(8/36),27%\n(14/52),53%\n(19/36),55%\n(11/20),2/6,1/2,28,15,14,15,72%\n(13/18),43%\n(9/21),65%\n(47/72),63%\n(45/72),38%\n(27/72),35%\n(25/72),51%\n(74/144),49%\n(70/144),91%\n(10/11),80%\n(8/10),20%\n(2/10),9%\n(1/11),57%\n(12/21),43%\n(9/21),alcaraz-garfia-carlos,sinner-jannik,18.03.2023 23:00,2\n-\n0\nFINISHED,indian-wells-2023,,
2,9,3,2,1,59%,59%,80%\n(35/44),70%\n(31/44),65%\n(20/31),48%\n(15/31),1/3,8/11,30%\n(13/44),20%\n(9/44),52%\n(16/31),35%\n(11/31),3/11,2/3,30,33,9,23,39%\n(7/18),64%\n(28/44),73%\n(55/75),61%\n(46/75),39%\n(29/75),27%\n(20/75),56%\n(84/150),44%\n(66/150),83%\n(10/12),75%\n(9/12),25%\n(3/12),17%\n(2/12),54%\n(13/24),46%\n(11/24),medvedev-daniil,tiafoe-frances,18.03.2023 20:45,2\n-\n0\nFINISHED,indian-wells-2023,,
3,2,6,2,6,62%,56%,81%\n(29/36),70%\n(32/46),59%\n(13/22),42%\n(15/36),3/4,9/12,30%\n(14/46),19%\n(7/36),58%\n(21/36),41%\n(9/22),3/12,1/4,26,24,8,13,69%\n(11/16),58%\n(11/19),72%\n(42/58),57%\n(47/82),43%\n(35/82),28%\n(16/58),55%\n(77/140),45%\n(63/140),90%\n(9/10),70%\n(7/10),30%\n(3/10),10%\n(1/10),60%\n(12/20),40%\n(8/20),alcaraz-garfia-carlos,auger-aliassime-felix,17.03.2023 01:35,2\n-\n0\nFINISHED,indian-wells-2023,,
4,4,5,0,1,54%,64%,70%\n(35/50),80%\n(45/56),55%\n(23/42),47%\n(15/32),6/9,1/3,20%\n(11/56),30%\n(15/50),53%\n(17/32),45%\n(19/42),2/3,3/9,25,32,17,17,50%\n(6/12),83%\n(15/18),63%\n(58/92),68%\n(60/88),32%\n(28/88),37%\n(34/92),48%\n(86/180),52%\n(94/180),80%\n(12/15),87%\n(13/15),13%\n(2/15),20%\n(3/15),47%\n(14/30),53%\n(16/30),fritz-taylor,sinner-jannik,16.03.2023 22:55,1\n-\n2\nFINISHED,indian-wells-2023,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,4,0,3,2,66%,53%,71%\n(35/49),67%\n(16/24),56%\n(14/25),62%\n(13/21),5/5,0/2,33%\n(8/24),29%\n(14/49),38%\n(8/21),44%\n(11/25),2/2,0/5,21,8,20,17,68%\n(15/22),50%\n(6/12),66%\n(49/74),64%\n(29/45),36%\n(16/45),34%\n(25/74),55%\n(65/119),45%\n(54/119),100%\n(10/10),78%\n(7/9),22%\n(2/9),0%\n(0/10),63%\n(12/19),37%\n(7/19),albot-radu,bellucci-mattia,06.03.2023 21:35,2\n-\n0\nFINISHED,indian-wells-2023,,
127,18,2,3,1,68%,60%,79%\n(45/57),75%\n(47/63),37%\n(10/27),52%\n(22/42),3/6,8/10,25%\n(16/63),21%\n(12/57),48%\n(20/42),63%\n(17/27),2/10,3/6,36,14,36,20,65%\n(15/23),54%\n(7/13),65%\n(55/84),66%\n(69/105),34%\n(36/105),35%\n(29/84),48%\n(91/189),52%\n(98/189),80%\n(12/15),88%\n(14/16),13%\n(2/16),20%\n(3/15),45%\n(14/31),55%\n(17/31),eubanks-christopher,marterer-maximilian,06.03.2023 19:10,1\n-\n2\nFINISHED,indian-wells-2023,,
12